# Semantic Response Caching

Exact-match caching only helps when users ask the identical question twice. Semantic caching goes further: if a new query is semantically similar to a cached one (e.g. "What is Python?" vs "Tell me about Python"), the cache returns the existing answer without calling the LLM. This cuts costs and latency dramatically for applications with overlapping question patterns.

**What you'll build:** An LLM pipeline with semantic caching backed by SQLite (local) and Redis (production), with cache hit/miss rate logging and a configurable similarity threshold.

**Time:** ~15 min | **Difficulty:** Beginner

**What you'll learn:**
- Configure `cache_backend` in `LLMConfig` to enable semantic caching
- Use the built-in `SQLiteCache` for development and local testing
- Switch to `RedisCache` for production without changing application code
- Tune the `similarity_threshold` to control how aggressively the cache matches
- Inspect `CacheStats` to measure hit rates and cost savings

## Prerequisites & Setup

In [ ]:
!pip install "synapsekit[openai,cache]" -q
# For Redis backend (optional): pip install "synapsekit[openai,cache,redis]" -q

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Set up a SQLite cache backend

`SQLiteCache` stores embeddings and cached responses in a local `.db` file. It computes a semantic embedding for each query and returns the cached response if the cosine similarity exceeds the threshold.

In [ ]:
import asyncio
from synapsekit import LLMConfig
from synapsekit.llms.openai import OpenAILLM
from synapsekit.cache import SQLiteCache, CacheStats

# SQLiteCache stores embeddings and cached responses in a local .db file.
cache = SQLiteCache(
    path="./llm_cache.db",
    similarity_threshold=0.92,   # 0 = cache everything, 1 = only exact matches
    max_entries=10_000,           # Evict oldest entries beyond this limit (LRU)
    ttl_seconds=86_400,           # Expire entries after 24 hours
)

# Pass the cache backend through LLMConfig -- the LLM client checks it on every call.
config = LLMConfig(
    temperature=0.2,
    max_tokens=512,
    cache_backend=cache,
)

llm = OpenAILLM(model="gpt-4o-mini", config=config)

print("SQLite cache configured with similarity_threshold=0.92")

## Step 2: Make cached calls and observe hits

The first call to the API stores the result. Semantically similar follow-up questions return the cached response immediately.

In [ ]:
async def demo_cache():
    stats = CacheStats()

    # First call -- no cache entry yet, calls the API
    q1 = "What is Python?"
    print(f"Query 1: {q1!r}")
    r1 = await llm.agenerate(q1)
    stats.record_miss()
    print(f"  Answer: {r1.text[:80]}...")
    print(f"  Source: API call (cache miss)\n")

    # Semantically similar -- should hit the cache
    q2 = "Tell me about the Python programming language."
    print(f"Query 2: {q2!r}")
    r2 = await llm.agenerate(q2)
    if r2.cache_hit:
        stats.record_hit()
        print(f"  Answer: {r2.text[:80]}...")
        print(f"  Source: CACHE HIT (saved ~$0.0001)\n")
    else:
        stats.record_miss()
        print(f"  Answer: {r2.text[:80]}...")
        print(f"  Source: API call (similarity below threshold)\n")

    # Clearly unrelated -- should miss
    q3 = "What is the boiling point of water?"
    print(f"Query 3: {q3!r}")
    r3 = await llm.agenerate(q3)
    stats.record_miss()
    print(f"  Answer: {r3.text[:80]}...")
    print(f"  Source: API call (cache miss)\n")

    print(f"Cache stats: {stats.hit_rate:.0%} hit rate  "
          f"({stats.hits} hits / {stats.total} total)")

await demo_cache()

## Step 3: Switch to Redis for production

`RedisCache` is a drop-in replacement for `SQLiteCache`. It stores embeddings in Redis with native TTL support.

In [ ]:
from synapsekit.cache import RedisCache

# RedisCache is a drop-in replacement for SQLiteCache.
redis_cache = RedisCache(
    url="redis://localhost:6379/0",
    similarity_threshold=0.92,
    ttl_seconds=3600,            # 1 hour TTL
    key_prefix="synapsekit:",    # Namespace keys to avoid collisions with other apps
)

# Swap the backend -- nothing else changes
prod_config = LLMConfig(
    temperature=0.2,
    max_tokens=512,
    cache_backend=redis_cache,
)

prod_llm = OpenAILLM(model="gpt-4o-mini", config=prod_config)

print("Redis cache backend configured (requires Redis running on localhost:6379)")

## Step 4: Pre-populate the cache (warm-up)

Pre-generate answers for your most common questions at startup so the first users after a deploy get cache hits rather than cold API calls with higher latency.

In [ ]:
async def warm_cache(common_questions: list[str]):
    """Pre-generate answers for your most common questions at startup."""
    print(f"Warming cache with {len(common_questions)} questions...")

    for question in common_questions:
        response = await llm.agenerate(question)
        print(f"  Cached: {question[:60]!r}")

    print("Cache warm-up complete.\n")

COMMON_QUESTIONS = [
    "What is machine learning?",
    "How does gradient descent work?",
    "What is a neural network?",
    "Explain backpropagation.",
    "What is overfitting?",
]

await warm_cache(COMMON_QUESTIONS)

## Complete Working Example

A self-contained demo that demonstrates MISS and HIT patterns across semantically similar queries.

In [ ]:
import asyncio
from synapsekit import LLMConfig
from synapsekit.llms.openai import OpenAILLM
from synapsekit.cache import SQLiteCache, CacheStats

async def main():
    cache = SQLiteCache(
        path="./demo_cache.db",
        similarity_threshold=0.92,
        ttl_seconds=86_400,
    )

    llm = OpenAILLM(
        model="gpt-4o-mini",
        config=LLMConfig(temperature=0.2, max_tokens=256, cache_backend=cache),
    )

    queries = [
        # (query, expected_source)
        ("What is a hash map?",                          "miss"),
        ("Explain hash tables in computer science.",     "hit"),   # Semantically similar
        ("How does a dictionary work in Python?",        "hit"),   # Also similar
        ("What is the speed of light?",                  "miss"),  # Unrelated
        ("How fast does light travel?",                  "hit"),   # Similar to above
    ]

    stats = CacheStats()

    print("=== Semantic Cache Demo ===\n")
    for query, expected in queries:
        response = await llm.agenerate(query)
        source = "HIT " if response.cache_hit else "MISS"

        if response.cache_hit:
            stats.record_hit()
        else:
            stats.record_miss()

        print(f"[{source}] {query!r}")
        print(f"       {response.text[:90]}...\n")

    print(f"Hit rate: {stats.hit_rate:.0%}  "
          f"({stats.hits}/{stats.total} queries served from cache)")
    print(f"Estimated savings: ${stats.hits * 0.0001:.4f}")

asyncio.run(main())

## Variations

In [ ]:
# Disable cache for specific calls (e.g. real-time data):
# response = await llm.agenerate(
#     "What is today's stock price for AAPL?",
#     cache_bypass=True,
# )

# Per-user cache isolation:
# user_cache = RedisCache(
#     url="redis://localhost:6379/0",
#     key_prefix=f"user:{user_id}:",
# )

# Cache with custom TTL per query:
# response = await llm.agenerate(prompt, cache_ttl=300)  # 5 minutes only

print("Variation examples shown above as comments.")

## Next Steps

- **[Cost-Aware LLM Router](cost-router.ipynb)** — complement caching with model-level cost routing
- **[LLM Fallback Chains](fallback-chain.ipynb)** — cache sits in front of the fallback chain for maximum resilience
- **[Structured Output with Pydantic](structured-output-pydantic.ipynb)** — cache structured responses alongside plain text

**Troubleshooting:**
- Cache is always missing for identical queries: Check that `similarity_threshold` is not above 0.99 (floating-point rounding)
- Redis connection refused: Start Redis with `redis-server --daemonize yes`
- Cache entries not expiring: Call `cache.evict_expired()` on a schedule — SQLite TTL is enforced lazily